### Generate the Black-Scholes Option Training Set

This notebook generates the training data for option pricing using Black-Scholes that will be used in a number of different examples. The [QuantLib](https://pypi.org/project/QuantLib-Python/) open source quantitative analytics library is used to value options and provide analytic sensitivities. This is not supplied as standard in Google Colab and so we must first install it.

In [ ]:
import sys
!{sys.executable} -m pip install --break-system-packages quantlib

Now we can import QuantLib and the other libraries that wil be needed.

In [ ]:
import QuantLib as ql
import math
import pandas as pd
import random as rd

We will generate 1 million samples which may sound a lot but the data will generate quickly.

In [ ]:
samples = 1000000

In [ ]:
option_data = list()

### Generate the dataset

Now we loop over the samples and generate the option prices. The option strike is fixed at 100.0 and the discount factor is set to be 1.0. This does not mean any loss of information as the discounting can be applied later and it is the relative value of strike and spot price that drives the model value.

The implied volatility, spot price and maturity are all generated randomly: 
* The implied volatility is assumed to follow a uniform distribution with values between 0% and 100%, $\sigma \sim U(0, 1)$.
* The spot price is assumed to be log-normal with the spot given by $100 e^z$ where $z \sim N(0.5, 0.25)$.
* The maturity follows a uniform distribution that is squared, in order to give more observations close to the option maturity. The formula used to generate the maturities is $T = t^2 / 365$ where $t \sim U(1, 43)$. The maturities lie approximately in the range 0 to 5 years. 

Note that all three parameters are generated independently and do not follow a specific stochastic model. This is quite deliberate and reflects the fact the aim here is to simply randomly sample the space of the input parameters and ensure reasonable coverage. 

Once the random input parameters have been generated these are fed into the Black-Scholes model from QuantLib. Calls and Puts are priced simultaneously as well as obtaining the main "Greeks", the delta, gamma, theta, vega and rho.

In [ ]:
for i in range(samples):
    
    sample_data = list()
    
    vol = rd.uniform(0.0, 1.0)
    sample_data.append(vol)
    spot = 100 * rd.lognormvariate(0.5, 0.25)
    sample_data.append(spot)
    T = rd.uniform(1,43)**2.0 / 365.0
    sample_data.append(T)
    
    call = ql.Option.Call
    put = ql.Option.Put
    strike = 100.0
    discount = 1.0
    
    strike_call = ql.PlainVanillaPayoff(call, strike)
    strike_put = ql.PlainVanillaPayoff(put, strike)
    
    stddev = vol * math.sqrt(T)
    
    black_call = ql.BlackCalculator(strike_call, 
                                    spot, 
                                    stddev, 
                                    discount)
    
    black_call_value = black_call.value()
    black_call_delta = black_call.delta(spot) 
    black_call_gamma = black_call.gamma(spot)
    black_call_theta = black_call.theta(spot, T)
    black_call_vega = black_call.vega(T)
    black_call_rho = black_call.rho(T)
    
    sample_data.append(black_call_value)
    sample_data.append(black_call_delta)
    sample_data.append(black_call_gamma)
    sample_data.append(black_call_theta)
    sample_data.append(black_call_vega)
    sample_data.append(black_call_rho)
     
    black_put = ql.BlackCalculator(strike_put, 
                                    spot, 
                                    stddev, 
                                    discount)
    
    black_put_value = black_put.value()
    black_put_delta = black_put.delta(spot) 
    black_put_gamma = black_put.gamma(spot)
    black_put_theta = black_put.theta(spot, T)
    black_put_vega = black_put.vega(T)
    black_put_rho = black_put.rho(T)
    
    sample_data.append(black_put_value)
    sample_data.append(black_put_delta)
    sample_data.append(black_put_gamma)
    sample_data.append(black_put_theta)
    sample_data.append(black_put_vega)
    sample_data.append(black_put_rho)
    
    option_data.append(sample_data)

Now the dataset has been generated the list of outputs is converted to a Pandas dataframe and the saved to a csv file for later use.

In [ ]:
heading_list = ['Vol', 'Spot', 'T', 'CallValue', 'CallDelta', 
                'CallGamma', 'CallTheta', 'CallVega', 
                'CallRho', 'PutValue', 'PutDelta', 
                'PutGamma', 'PutTheta', 'PutVega', 'PutRho']

In [ ]:
df = pd.DataFrame(option_data, columns=heading_list)

In [ ]:
df.head()

In [ ]:
df.to_csv('OptionData.csv', index=False)

### Visualise the Input Parameter Distributions

It is useful to visualise the generated distributions for the input parameters, the implied volatility, spot price and maturity. This can done by plotting histograms of the generated data using matplotlib.

In [ ]:
%matplotlib inline
import matplotlib.mlab as mlab
import matplotlib.pyplot as plt

#### Volatility

In [ ]:
# the histogram of the vol
n, bins, patches = plt.hist(df.Vol, 50, density=True, facecolor='k')

plt.xlabel('Volatility')
plt.ylabel('Density')
plt.grid(True)
plt.savefig('BlackScholesVol.png', dpi=300, bbox_inches='tight')

#### Spot Price

In [ ]:
# the histogram of the spot
n, bins, patches = plt.hist(df.Spot, 50, density=True, facecolor='k')

plt.xlabel('Forward')
plt.ylabel('Density')
plt.grid(True)
plt.savefig('BlackScholesFwd.png', dpi=300, bbox_inches='tight')

#### Maturity

In [ ]:
# the histogram of the maturity
n, bins, patches = plt.hist(df['T'], 50, density=True, facecolor='k')

plt.xlabel('Maturity')
plt.ylabel('Density')
plt.grid(True)
plt.savefig('BlackScholesT.png', dpi=300, bbox_inches='tight')